# 03 — Retrieval Evaluation

Ecommerce KPI RAG Capstone — LLM Zoomcamp 2026

We compare **text search** (minsearch, keyword-based) against **vector search**
(sentence-transformers embeddings) using Hit Rate and Mean Reciprocal Rank (MRR) at k=5.

**Ground truth:** since each KPI document answers a specific, narrow question (e.g. "revenue in
the Electronics category"), we can generate template questions per document and know exactly
which document ID is the correct retrieval target — no manual labeling needed.

## 1. Load documents

In [1]:
import json

with open("../data/documents.json") as f:
    documents = json.load(f)

print(f"{len(documents)} documents loaded")

59 documents loaded


## 2. Generate ground-truth evaluation questions

In [2]:
QUESTION_TEMPLATES = {
    "category": [
        "How much revenue did the {category} category generate?",
        "What is the average order value for {category}?",
    ],
    "country": [
        "What is the total revenue in {country}?",
        "How many customers do we have in {country}?",
    ],
    "loyalty_tier": [
        "How much revenue comes from {loyalty_tier} tier customers?",
    ],
    "month": [
        "What was the revenue in {month}?",
    ],
    "channel": [
        "How much revenue came from the {channel} marketing channel?",
    ],
    "overall": [
        "What is the total revenue across all orders?",
    ],
}

ground_truth = []
for doc in documents:
    templates = QUESTION_TEMPLATES.get(doc["type"], [])
    for template in templates:
        question = template.format(**doc)
        ground_truth.append({"question": question, "doc_id": doc["id"]})

print(f"{len(ground_truth)} evaluation questions generated")
ground_truth[:5]

72 evaluation questions generated


[{'question': 'How much revenue did the Beauty category generate?',
  'doc_id': 0},
 {'question': 'What is the average order value for Beauty?', 'doc_id': 0},
 {'question': 'How much revenue did the Electronics category generate?',
  'doc_id': 1},
 {'question': 'What is the average order value for Electronics?', 'doc_id': 1},
 {'question': 'How much revenue did the Fashion category generate?',
  'doc_id': 2}]

Running this against the 59 documents produces **72 evaluation questions** — 1-2 per document depending on type.

In [7]:
with open("../data/eval_questions.json", "w") as f:
    json.dump(ground_truth, f, indent=2)

## 3. Evaluation metrics

- **Hit Rate@k** — fraction of questions where the correct document appears anywhere in the top-k results
- **MRR@k** — mean of 1/rank of the correct document (0 if not in top-k); rewards ranking the right answer *first*, not just somewhere in the list

In [4]:
def hit_rate(results_list, k=5):
    hits = 0
    for correct_id, results in results_list:
        retrieved_ids = [r["id"] for r in results[:k]]
        if correct_id in retrieved_ids:
            hits += 1
    return hits / len(results_list)

def mrr(results_list, k=5):
    total = 0.0
    for correct_id, results in results_list:
        retrieved_ids = [r["id"] for r in results[:k]]
        if correct_id in retrieved_ids:
            rank = retrieved_ids.index(correct_id) + 1
            total += 1.0 / rank
    return total / len(results_list)

## 4. Run both retrieval approaches

Uses the shared indexing module (`src/rag_index.py`) so this notebook and Notebook 02 test
against the exact same text and vector indexes — add `src/` to your path first.

In [9]:
import sys
sys.path.insert(0, "../src")
from rag_index import text_search, vector_search


In [10]:

K = 6
text_results = [(gt["doc_id"], text_search(gt["question"], num_results=K)) for gt in ground_truth]
vector_results = [(gt["doc_id"], vector_search(gt["question"], num_results=K)) for gt in ground_truth]

In [11]:
print("Text search  (minsearch):")
print(f"  Hit Rate@{K}: {hit_rate(text_results, K):.3f}")
print(f"  MRR@{K}:      {mrr(text_results, K):.3f}")

print("\nVector search (sentence-transformers):")
print(f"  Hit Rate@{K}: {hit_rate(vector_results, K):.3f}")
print(f"  MRR@{K}:      {mrr(vector_results, K):.3f}")

Text search  (minsearch):
  Hit Rate@6: 1.000
  MRR@6:      0.859

Vector search (sentence-transformers):
  Hit Rate@6: 0.764
  MRR@6:      0.632


## 5. Record results

Fill in the actual numbers after running Section 4 locally, and copy this table into the README.

| Approach | Hit Rate@6 | MRR@6 |
|---|------------|-------|
| Text search (minsearch) | 1.000      | 0.859 |
| Vector search (sentence-transformers) | 0.764      | 0.632 |

**Expectation going in:** since our questions closely echo the exact wording in the documents
(e.g. "revenue in the Electronics category" vs. document text "Category: Electronics. Total
revenue..."), text search should do very well here — this is a favorable case for keyword
matching. Vector search should still perform competitively since the embeddings capture the
same semantic content, but the gap (if any) is informative: if text search wins clearly, it
tells us our questions are close to a keyword-search-friendly domain (structured KPI data),
not a reason to abandon vector search generally.